**Black-Litterman**

- Combine market-implied expected returns with an uncertain investor view.

- Germany-versus-Europe example from the course slides. Inputs are illustrative and annual; returns are excess returns in a common currency.

### TODO 📝

Replace each `None` marked with `# TODO` with your implementation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Market prior

Market capitalizations determine the benchmark weights $w_M$. 

We infer the expected returns for which the market portfolio is mean–variance optimal:

$$
    \pi=\delta\Sigma w_M, \qquad \mu\sim\mathcal N(\pi,C), \qquad C=\tau\Sigma
$$

- $\Sigma$ is return covariance; 
- $C$ is uncertainty about expected returns;
- $\delta$ is risk aversion. 

> Smaller $\tau$ means more confidence in the market prior (investors views are less important). 

> Larger $\tau$ means more uncertainty about the market prior (investors views are more important).

In [ ]:
assets = ["DAX", "UKX", "CAC40", "AEX"]  # Germany, UK, France, Netherlands
market_caps = np.array([2.0, 3.0, 1.5, 0.5])
w_market = None  # TODO: Normalize market capitalizations into benchmark weights.
vol = np.array([0.20, 0.18, 0.22, 0.21]).reshape(-1, 1)  # volatilities of the assets
corr = np.array(
    [
        [1.00, 0.65, 0.80, 0.75],
        [0.65, 1.00, 0.60, 0.60],
        [0.80, 0.60, 1.00, 0.75],
        [0.75, 0.60, 0.75, 1.00],
    ]
)
Sigma = (
    None  # TODO: Build the (4, 4) covariance matrix from volatilities and correlations.
)
risk_aversion = 2.5
tau = 0.05
pi = None  # TODO: Compute market-implied expected returns by reverse optimization.
C = tau * Sigma

## Investor view

Germany outperforms a cap-weighted basket of the UK, France, and Netherlands by 5 percentage points:

$$
    q=P\mu+\varepsilon, \qquad
    P=\begin{bmatrix}1 & -0.60 & -0.30 & -0.10\end{bmatrix}, \qquad
    \varepsilon\sim\mathcal N(0,\Omega)
$$

Set $q=0.05$ and $\Omega=[0.05^2]$. The 5 percentage-point standard deviation measures uncertainty in the expected spread. Smaller $\Omega$ means more confidence.

In [ ]:
# Germany is excluded from the mkt portfolio.
rest_weights = market_caps[1:] / market_caps[1:].sum()
P = None  # TODO: Build a (1, 4) view matrix: Germany minus the cap-weighted rest of Europe.
q = np.array([0.05])
view_sd = 0.05
Omega = np.array([[view_sd**2]])

## Posterior expected returns

$$
    K=CP^\top(PCP^\top+\Omega)^{-1}, \qquad
    \mu_{BL}=\pi+K(q-P\pi)
$$

The prior is adjusted by the difference between the view and the prior spread. 

Change `view_sd` above to 0.10 or 0.005, then compare weaker and stronger confidence in the view.

In [ ]:
# spread = Germany's expected return minus the weighted average of the other
# countries' expected returns
CPt = C @ P.T
K = None  # TODO: Compute the (4, 1) gain matrix using np.linalg.solve.
mu_bl = None  # TODO: Update the prior using the difference between the view and prior spread.

print(f"Prior spread:     {(P @ pi).item():.2%}")
print(f"Investor view:    {q.item():.2%}")
print(f"Posterior spread: {(P @ mu_bl).item():.2%}")

## Portfolio weights

Keep return covariance fixed at $\Sigma$:

$$
    w_{BL}=\frac{1}{\delta}\Sigma^{-1}\mu_{BL}
$$

The allocation is unconstrained: short positions and weights above 100% are allowed. For this relative view, the adjustment sums to zero, so weights still sum to one.

In [ ]:
# TODO: Solve for the unconstrained portfolio weights using np.linalg.solve.
w_bl = None
comparison = pd.DataFrame(
    {
        "Prior mean (%)": 100 * pi,
        "BL mean (%)": 100 * mu_bl,
        "Market weight (%)": 100 * w_market,
        "BL weight (%)": 100 * w_bl,
    },
    index=assets,
)
comparison.round(2)

In [ ]:
ax = comparison[["Market weight (%)", "BL weight (%)"]].plot.bar(
    figsize=(8, 4), rot=0, color=["#64748b", "#2563eb"]
)
ax.axhline(0, color="black", lw=0.7)
ax.set_ylabel("Portfolio weight (%)")
ax.legend(handles=ax.containers, labels=["Market", "Black-Litterman"])
plt.tight_layout()
plt.show()